# CREM: Renta Neta y Bruta Media por Persona según Municipios
**Propósito**: Extrae la información sobre la renta neta y bruta media por persona por municipios a partir de la página oficial del CREM.

**Particularidades de esta tabla**:
- La cabecera tiene **dos filas**: la primera agrupa sub-tablas por nombre (con `colspan`), la segunda contiene los años.
- El nombre de columna resultante es `{año}_{nombre_subtabla}` normalizado.
- Se filtran únicamente las filas de nivel **Municipio** (`level2`).

In [1]:
import sys
import re
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
url_base   = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM2089//sec2.html"
table_name = "raw.crem.renta_neta_y_bruta_media_por_persona"

## Descarga y Parseo del Contenido HTML directamente en Memoria

In [3]:
html = crem.get_html(url_base)
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")
if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# ---------------------------------------------------------------------------
# Parseo de cabecera de dos filas:
#   Fila 0: grupos con colspan  → nombre del grupo (sub-tabla)
#   Fila 1: años                → valor de cada columna de datos
# Resultado: lista de etiquetas compuestas  "{año}_{Grupo}"
# ---------------------------------------------------------------------------
thead    = table.find("thead", id="cabeceraTablaDatos") or table.find("thead")
hdr_rows = thead.find_all("tr")

# Fila 0: grupos con colspan
grupos = []
for th in hdr_rows[0].find_all("th"):
    texto = th.get_text(strip=True)
    if not texto:
        continue
    span = int(th.get("colspan", 1))
    grupos.extend([texto] * span)

# Fila 1: años
anios = [th.get_text(strip=True) for th in hdr_rows[1].find_all("th") if th.get_text(strip=True)]

columnas = [f"{a}_{g}" for a, g in zip(anios, grupos)]
print(f"Columnas detectadas ({len(columnas)}): {columnas[:6]} …")

Columnas detectadas (18): ['2023_Renta neta media por persona', '2022_Renta neta media por persona', '2021_Renta neta media por persona', '2020_Renta neta media por persona', '2019_Renta neta media por persona', '2018_Renta neta media por persona'] …


In [4]:
# ---------------------------------------------------------------------------
# Parseo de filas: solo level2 (Municipio)
# ---------------------------------------------------------------------------
tbody = table.find("tbody")

mapping_niveles = {1: "Región", 2: "Municipio", 3: "Distrito", 4: "Sección"}

datos_filas = []
for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue

    # Detectar nivel
    ul = th.find("ul")
    level_num = None
    if ul and ul.has_attr("class"):
        for c in ul["class"]:
            if c.startswith("level"):
                try:
                    level_num = int(c.replace("level", ""))
                except ValueError:
                    pass
                break

    # Solo Municipios
    if mapping_niveles.get(level_num) != "Municipio":
        continue

    label_clean = crem.clean_and_decode(th.get_text(strip=True))
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    nombre = match.group(1).strip() if match else label_clean
    if ',' in nombre:
        nombre = nombre.split(',')[1].strip() + ' ' + nombre.split(',')[0].strip()

    tds    = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            values.append(float(val_str) if val_str not in ("", "-", "..") else None)
        except ValueError:
            values.append(None)

    if len(values) == len(columnas):
        registro = {"nombre": crem.normalize_name(nombre)}
        for col, val in zip(columnas, values):
            registro[col] = val
        datos_filas.append(registro)

print(f"Municipios parseados: {len(datos_filas)}")

Municipios parseados: 45


## Integración con PySpark y Delta Lake

In [5]:
spark = tfm_utils.get_spark_session(app_name="CREM_Renta_Neta_Bruta_Persona")

df = spark.createDataFrame(datos_filas)

# Normalizar columnas (minúsculas, sin acentos)
df_normalized = tfm_utils.normalize_df(df)
tfm_utils.display(df_normalized)

,2015_renta_bruta_media_por_persona,2015_renta_neta_media_por_persona,2016_renta_bruta_media_por_persona,2016_renta_neta_media_por_persona,2017_renta_bruta_media_por_persona,2017_renta_neta_media_por_persona,2018_renta_bruta_media_por_persona,2018_renta_neta_media_por_persona,2019_renta_bruta_media_por_persona,2019_renta_neta_media_por_persona,2020_renta_bruta_media_por_persona,2020_renta_neta_media_por_persona,2021_renta_bruta_media_por_persona,2021_renta_neta_media_por_persona,2022_renta_bruta_media_por_persona,2022_renta_neta_media_por_persona,2023_renta_bruta_media_por_persona,2023_renta_neta_media_por_persona,nombre
0,9.303,8.274,9.478,8.444,10.235,9.044,10.617,9.415,11.168,9.860,11.385,10.021,11.856,10.331,12.751,11.124,13.954,12.104,abanilla
1,9.397,8.338,9.495,8.401,10.063,8.845,10.540,9.283,12.822,10.683,12.076,10.343,11.783,10.212,12.323,10.717,13.185,11.501,abaran
2,9.093,7.977,9.469,8.269,10.028,8.684,10.381,9.011,10.861,9.383,11.133,9.658,11.681,10.110,12.434,10.556,13.421,11.511,aguilas
3,8.073,7.458,8.332,7.680,9.076,8.340,9.298,8.539,9.703,8.899,10.319,9.409,10.502,9.361,10.983,9.901,12.010,10.789,albudeite
4,9.554,8.305,9.747,8.465,10.254,8.835,10.686,9.197,11.272,9.706,11.486,9.885,12.046,10.312,12.673,10.805,13.668,11.621,alcantarilla
5,10.443,9.322,10.535,9.357,10.790,9.551,11.460,10.141,12.197,10.763,12.265,10.837,12.896,10.921,13.304,11.600,14.366,12.499,aledo
6,8.385,7.421,8.613,7.605,9.174,8.054,9.489,8.342,9.877,8.683,10.051,8.836,10.487,9.168,10.963,9.533,11.762,10.246,alguazas
7,10.587,9.103,10.789,9.218,11.296,9.650,11.674,9.888,12.365,10.526,12.828,10.654,13.132,10.943,13.778,11.637,14.670,12.394,alhama_de_murcia
8,8.873,7.837,9.091,8.006,9.609,8.434,9.950,8.728,10.493,9.174,10.833,9.443,11.472,9.833,12.035,10.385,12.751,10.990,archena
9,7.719,6.899,8.022,7.139,8.722,7.675,9.025,7.935,9.484,8.350,9.704,8.550,10.103,8.867,10.714,9.377,11.555,10.108,beniel


In [6]:
delta_path = tfm_utils.full_table_path(table_name)
print(f"Escribiendo {df_normalized.count()} registros en: {table_name}")
print(f"Destino: {delta_path}")

(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)
print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en: raw.crem.renta_neta_y_bruta_media_por_persona
Destino: /tfm/delta_lake/raw/crem/renta_neta_y_bruta_media_por_persona
¡Escritura en Delta Lake completada con éxito!
